In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import  train_test_split
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

import matplotlib.pylab as plt


In [4]:
file = "DigitalLifestyle.csv"
df = pd.read_csv(file)

# I will be using all numeric data and dropping the categorical.The features used could be changed to less number of more correlatated options
# to make it more practical.
target = df['high_risk_flag']
features = df.drop(['id', 'high_risk_flag', 'device_type', 'gender','region','income_level','education_level','daily_role'], axis=1)

# Randomly split data into training and testing data(x_train and y_train are pairs and x_test and y_test are pairs)
x_train, x_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=42)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

((2800, 16), (700, 16), (2800,), (700,))

In [5]:
features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 16 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       3500 non-null   int64  
 1   device_hours_per_day      3500 non-null   float64
 2   phone_unlocks             3500 non-null   int64  
 3   notifications_per_day     3500 non-null   int64  
 4   social_media_mins         3500 non-null   int64  
 5   study_mins                3500 non-null   int64  
 6   physical_activity_days    3500 non-null   float64
 7   sleep_hours               3500 non-null   float64
 8   sleep_quality             3500 non-null   float64
 9   anxiety_score             3500 non-null   float64
 10  depression_score          3500 non-null   float64
 11  stress_level              3500 non-null   float64
 12  happiness_score           3500 non-null   float64
 13  focus_score               3500 non-null   float64
 14  producti

In [6]:
train_data = lgb.Dataset(x_train, label=y_train, free_raw_data=False)
test_data = lgb.Dataset(x_test, label=y_test, free_raw_data=False)

Training model

In [7]:
params = {'objective':'binary', #binary because our dependent variable is either 1 or 0, its classifying into either category.
          'metric':'auc', #metric used for evaluation
          'boosting':'gbdt', #Specifying our model is a gradient boosting decision tree.
          'num_leaves': 60, # num of leaf nodes on trees created by model
          'min_data_in_leaf': 6,
          'feature_fraction': 0.70, # specifying the number of features each tree will consider(example: 0.5 would be 50%). This also helps with speed too
          'bagging_fraction': 0.5, # This fraction of this previous tree is used to create the next tree
          'bagging_frequency': 2, # Every n number of iterations, lightGBM will randomly select the specified fraction of observations and use them for the next 20 iterations
          'learning_rate': 0.03, 
          'verbose': -1 # controls number of logs printed while model trains
         }
          

In [15]:
from lightgbm import early_stopping

GBDT_model = lgb.train(params,
                      train_data,
                      valid_sets=test_data,
                      num_boost_round=5,
                      callbacks=[lgb.early_stopping(50, verbose=True)])
                    

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[2]	valid_0's auc: 0.78693


## Other Models

These are other boosting algorithms I wanted to try, didn't tweak hypr params though

In [9]:
# ------ Dropout Additive Regression Trees ------

params = {'objective':'binary', #binary because our dependent variable is either 1 or 0, its classifying into either category.
          'metric':'auc', #metric used for evaluation, lowkey not sure what this means though.
          'boosting':'dart', #Specifying our model is using Dropout Additive Regression Trees
          'num_leaves': 60, # num of leaf nodes on trees created by model
          'min_data_in_leaf': 6,
          'feature_fraction': 0.70, # specifying the number of features each tree will consider(example: 0.5 would be 50%). This also helps with speed too
          'bagging_fraction': 0.5, # not sure what this does yet. I think something about this fraction of observations will be used to create the next tree?
          'bagging_frequency': 2, # Every n number of iterations, lightGBM will randomly select the specified fraction of observations and use them for the next 20 iterations
          'learning_rate': 0.03, # this seems to be the sweet spot
          'verbose': -1 # controls number of logs printed while model trains
         }

DART_model = lgb.train(params,
                      train_data,
                      valid_sets=test_data,
                      num_boost_round=2, #
                      keep_training_booster=True,
                      callbacks=[lgb.log_evaluation(period=1)])

[1]	valid_0's auc: 0.727249
[2]	valid_0's auc: 0.78693


In [10]:
# ------ Gradient-based One-Side Sampling ------

params = {'objective':'binary', #binary because our dependent variable is either 1 or 0, its classifying into either category.
          'metric':'auc', #metric used for evaluation, lowkey not sure what this means though.
          'boosting':'goss', #Specifying our model is using Gradient-based One-Side Sampling
          'num_leaves': 60, # num of leaf nodes on trees created by model
          'min_data_in_leaf': 6,
          'feature_fraction': 0.70, # specifying the number of features each tree will consider(example: 0.5 would be 50%). This also helps with speed too
          'bagging_fraction': 0.5, # not sure what this does yet. I think something about this fraction of observations will be used to create the next tree?
          'bagging_frequency': 2, # Every n number of iterations, lightGBM will randomly select the specified fraction of observations and use them for the next 20 iterations
          'learning_rate': 0.03, # this seems to be the sweet spot
          'verbose': -1 # controls number of logs printed while model trains
         }

GOSS_model = lgb.train(params,
                      train_data,
                      valid_sets=test_data,
                      num_boost_round=3, #
                      keep_training_booster=True,
                      callbacks=[lgb.log_evaluation(period=1)])

[1]	valid_0's auc: 0.727249
[2]	valid_0's auc: 0.78693
[3]	valid_0's auc: 0.784119


## Here are the model metrics

In [16]:
y_train_pred = GBDT_model.predict(x_train)
y_test_pred = GBDT_model.predict(x_test)

print("-----Gradient Boosting Decision Tree-----")
print("AUC train: {:.4f}\nAUC Test: {:.4f}".format(roc_auc_score(y_train, y_train_pred),
                                                   roc_auc_score(y_test, y_test_pred)))
y_train_pred = DART_model.predict(x_train)
y_test_pred = DART_model.predict(x_test)

print("\n-----Dropout Additive Regression Tree-----")
print("AUC train: {:.4f}\nAUC Test: {:.4f}".format(roc_auc_score(y_train, y_train_pred),
                                                   roc_auc_score(y_test, y_test_pred)))
y_train_pred = GOSS_model.predict(x_train)
y_test_pred =  GOSS_model.predict(x_test)

print("\n-----Gradient Based One-Sided Sampling-----")
print("AUC train: {:.4f}\nAUC Test: {:.4f}".format(roc_auc_score(y_train, y_train_pred),
                                                   roc_auc_score(y_test, y_test_pred)))
                                                    

-----Gradient Boosting Decision Tree-----
AUC train: 0.8970
AUC Test: 0.7869

-----Dropout Additive Regression Tree-----
AUC train: 0.8970
AUC Test: 0.7869

-----Gradient Based One-Sided Sampling-----
AUC train: 0.8970
AUC Test: 0.7841
